In [ ]:
import numpy as np
import pynq

SAMPLES = 600
W = 32

overlay = pynq.Overlay("./fir11.bit")

print(overlay.ip_dict.keys())

In [ ]:
# 根据上一格实际显示的 IP 名修改
fir = overlay.fir_0

register_map = fir.register_map

print(register_map)

In [ ]:
with open("input.dat", "r") as f:
    x = [int(line.strip()) for line in f if line.strip()]

assert len(x) >= SAMPLES

x = x[:SAMPLES]

print("Number of samples:", len(x))
print("First 10 samples:", x[:10])

In [ ]:
def sign_extend(v, W):
    v &= (1 << W) - 1

    if v & (1 << (W - 1)):
        v -= (1 << W)

    return v


def fir_hw(signal):

    # Write x
    register_map.x = int(signal) & ((1 << W) - 1)

    # Start
    register_map.CTRL.AP_START = 1

    # Wait until done
    while register_map.CTRL.AP_DONE == 0:
        pass

    # Read output
    raw_y = int(register_map.y)

    # AP_CONTINUE, CTRL bit 4
    fir.write(0x00, 0x10)

    # Convert unsigned raw bits to signed Python int
    return sign_extend(raw_y, W)

In [ ]:
y = []

for i in range(SAMPLES):

    signal = x[i]

    output = fir_hw(signal)

    y.append(output)

    print(i, signal, output)

In [ ]:
with open("out.dat", "w") as f:
    for value in y:
        f.write(f"{value}\n")

print("out.dat written.")

In [ ]:
with open("out.gold.dat", "r") as f:
    golden = [int(line.strip()) for line in f if line.strip()]

assert len(golden) >= SAMPLES

golden = golden[:SAMPLES]

In [ ]:
errors = []

for i in range(SAMPLES):
    if y[i] != golden[i]:
        errors.append((i, x[i], y[i], golden[i]))


if len(errors) == 0:
    print("*********************************************")
    print("PASS: The output matches the golden output!")
    print("*********************************************")

else:
    print("*********************************************")
    print("FAIL: Output DOES NOT match the golden output")
    print("*********************************************")

    print(f"Number of mismatches: {len(errors)}")

    print("\nFirst 20 mismatches:")
    print("index  input  hardware  golden")

    for i, signal, hw, gold in errors[:20]:
        print(f"{i:5d} {signal:6d} {hw:9d} {gold:7d}")